# Time and space resolved modeling

In the following the use of the constraints class for the definition of time points and sub_models is demonstrated
in an example by defining 6 time points and two sub_models. These defined changes are then applied to a Cobra model.

In [1]:
import cobra

from cobra import Reaction
from importlib_resources import files, as_file

from model_duplication.constraints.constraints import Constraints
from model_duplication.constraints.linker import Linker
from cobra.io import read_sbml_model

We create a new Cobra model and a constraints object.

In [2]:
textbook_raw = files(cobra.data).joinpath("textbook.xml.gz")
with as_file(textbook_raw) as textbookXML:
    textbook = read_sbml_model(str(textbookXML))

model = textbook.copy()
con = Constraints()

Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  1.000e+00  ratio =  1.000e+00
Problem data seem to be well scaled


As a default, a time phase with one hour and a sub-model with a volume of one are created. This is replaced by as soon
as explicit definitions of time phases and sub-models are made.

In [3]:
print(con.phases)

+---------+---------------+--------+-----------+
|  Phase  |      Name     | Volume | Timeframe |
+---------+---------------+--------+-----------+
| default | Default Phase |   1    |     1     |
+---------+---------------+--------+-----------+


We add two sub-models whose volume is 1 each.Furthermore, we have them displayed. It is important that the default
sub-model is deleted and replaced by the newly defined ones.

In [4]:
con.add_sub_models(
    labels=["leaf","root"],
    volumes=[1,1]
)
print(con.phases)

+--------+------+--------+-----------+
| Phase  | Name | Volume | Timeframe |
+--------+------+--------+-----------+
| leaf-0 |      |   1    |     1     |
| root-0 |      |   1    |     1     |
+--------+------+--------+-----------+


Now we add time phases. With these it is only possible to add n phases with the same time length. If different time
lengths are required, the function must be used several times.

In [5]:
con.add_time_slots(
    n_ranges=6,
    time=2,
    light_dark="light"
)
print(con.phases)

+--------+------+--------+-----------+
| Phase  | Name | Volume | Timeframe |
+--------+------+--------+-----------+
| leaf-0 |      |   1    |     2     |
| root-0 |      |   1    |     2     |
| leaf-1 |      |   1    |     2     |
| root-1 |      |   1    |     2     |
| leaf-2 |      |   1    |     2     |
| root-2 |      |   1    |     2     |
| leaf-3 |      |   1    |     2     |
| root-3 |      |   1    |     2     |
| leaf-4 |      |   1    |     2     |
| root-4 |      |   1    |     2     |
| leaf-5 |      |   1    |     2     |
| root-5 |      |   1    |     2     |
+--------+------+--------+-----------+


Here we now use the print method of Contraints instead of just displaying the phases as before.

In [6]:
print(con)

+----------------------+--------+--------+--------+--------+--------+--------+
| Sub-Model\Time Index |   0    |   1    |   2    |   3    |   4    |   5    |
+----------------------+--------+--------+--------+--------+--------+--------+
|          | id        | leaf-0 | leaf-1 | leaf-2 | leaf-3 | leaf-4 | leaf-5 |
|    leaf  | volume    |   1    |   1    |   1    |   1    |   1    |   1    |
|          | time      |   2    |   2    |   2    |   2    |   2    |   2    |
+----------------------+--------+--------+--------+--------+--------+--------+
|          | id        | root-0 | root-1 | root-2 | root-3 | root-4 | root-5 |
|    root  | volume    |   1    |   1    |   1    |   1    |   1    |   1    |
|          | time      |   2    |   2    |   2    |   2    |   2    |   2    |
+----------------------+--------+--------+--------+--------+--------+--------+


Linkers are defined below.

In [7]:
linker = Linker(
        source="leaf-0",
        destination="leaf-1",
        id="acald_e"
    )

con.add_linker(linker)

The previously defined adjustments are now applied to the model.

In [8]:
reaction = Reaction(
        id = "ATPM",
        lower_bound=456,
        upper_bound=765,
)

con.add_reaction_to_phase(reaction,"leaf-0")

In [9]:
new_model = con.apply_to_model(model)

New suffix 'leaf-0' for model added
Iteration 1 for 'root-0' completed
Iteration 2 for 'leaf-1' completed
Iteration 3 for 'root-1' completed
Iteration 4 for 'leaf-2' completed
Iteration 5 for 'root-2' completed
Iteration 6 for 'leaf-3' completed
Iteration 7 for 'root-3' completed
Iteration 8 for 'leaf-4' completed
Iteration 9 for 'root-4' completed
Iteration 10 for 'leaf-5' completed
Iteration 11 for 'root-5' completed
Model e_coli_core successfully modified


Since the model contains 95 reactions, we should get 95 * 6 * 2 + 1 = 1141 reactions.

In [10]:
print(len(model.reactions))

95


In [11]:
print(len(new_model.reactions))

1141


Now let's take a quick look at the defined adjustment of the reaction. Here we see that the original and the reaction in
phase leaf-1 have identical parameters. The reaction in leaf-0, however, has the parameters we defined earlier instead
of the original ones.

In [12]:
model.reactions.get_by_id("ATPM")

Reaction identifier,ATPM
Name,ATP maintenance requirement
Memory address,0x07fbdc45646a0
Stoichiometry,atp_c + h2o_c --> adp_c + h_c + pi_c ATP + H2O --> ADP + H+ + Phosphate
GPR,
Lower bound,8.39
Upper bound,1000.0


In [13]:
new_model.reactions.get_by_id("ATPM_leaf-0")

Reaction identifier,ATPM_leaf-0
Name,ATP maintenance requirement
Memory address,0x07fbdc4303a30
Stoichiometry,atp_c_leaf-0 + h2o_c_leaf-0 --> adp_c_leaf-0 + h_c_leaf-0 + pi_c_leaf-0 ATP + H2O --> ADP + H+ + Phosphate
GPR,
Lower bound,456
Upper bound,765


In [14]:
new_model.reactions.get_by_id("ATPM_leaf-1")

Reaction identifier,ATPM_leaf-1
Name,ATP maintenance requirement
Memory address,0x07fbdc406f280
Stoichiometry,atp_c_leaf-1 + h2o_c_leaf-1 --> adp_c_leaf-1 + h_c_leaf-1 + pi_c_leaf-1 ATP + H2O --> ADP + H+ + Phosphate
GPR,
Lower bound,8.39
Upper bound,1000.0


In the following the state of the Constraints object is saved as XML file. Based on this file a new Constraints object
can be created, which corresponds to the original one.

In [15]:
con.save_as_xml("./data/conf.xml")
load = Constraints.load_from_xml("./data/conf.xml")

Original

In [16]:
print(con)

+----------------------+--------+--------+--------+--------+--------+--------+
| Sub-Model\Time Index |   0    |   1    |   2    |   3    |   4    |   5    |
+----------------------+--------+--------+--------+--------+--------+--------+
|          | id        | leaf-0 | leaf-1 | leaf-2 | leaf-3 | leaf-4 | leaf-5 |
|    leaf  | volume    |   1    |   1    |   1    |   1    |   1    |   1    |
|          | time      |   2    |   2    |   2    |   2    |   2    |   2    |
+----------------------+--------+--------+--------+--------+--------+--------+
|          | id        | root-0 | root-1 | root-2 | root-3 | root-4 | root-5 |
|    root  | volume    |   1    |   1    |   1    |   1    |   1    |   1    |
|          | time      |   2    |   2    |   2    |   2    |   2    |   2    |
+----------------------+--------+--------+--------+--------+--------+--------+


Loaded version

In [17]:
print(load)

+----------------------+--------+--------+--------+--------+--------+--------+
| Sub-Model\Time Index |   0    |   1    |   2    |   3    |   4    |   5    |
+----------------------+--------+--------+--------+--------+--------+--------+
|          | id        | leaf-0 | leaf-1 | leaf-2 | leaf-3 | leaf-4 | leaf-5 |
|    leaf  | volume    |   1    |   1    |   1    |   1    |   1    |   1    |
|          | time      |   2    |   2    |   2    |   2    |   2    |   2    |
+----------------------+--------+--------+--------+--------+--------+--------+
|          | id        | root-0 | root-1 | root-2 | root-3 | root-4 | root-5 |
|    root  | volume    |   1    |   1    |   1    |   1    |   1    |   1    |
|          | time      |   2    |   2    |   2    |   2    |   2    |   2    |
+----------------------+--------+--------+--------+--------+--------+--------+
